# TPUForge Multi-Chip Kaggle TPU v5e-8, Colab TPU v5e-1, & Cloud TPU v6e/v7 Benchmark

This notebook builds the native C++20 `tpuforge` runtime (`_tpuforge_ext` + `tpuforge_bench`) and benchmarks **M-005 Optical Torus Ring-Attention (ICI)** plus **Vanilla JAX**, **StaticArena (`M-002`)**, and **TPUForge Tree Speculation (`M-002 + M-003`)** on real **Qwen2.5-Coder** weights.

### Hardware-Adaptive Kaggle `v5e-8`, Colab `v5e-1`, & Cloud TPU Support
- **8x TPU v5e chips (`128 GiB` pooled HBM):** Runs `128k` ICI Ring-Attention, `Qwen/Qwen2.5-Coder-7B-Instruct` (`TP=4`), and fits `Qwen/Qwen2.5-Coder-32B-Instruct` via **OpenXLA GSPMD Megatron Tensor Parallelism (`TP=8`)** + streaming shard-by-shard `.safetensors` loading (`~2.3 GB` peak host RAM per weight stack).
- **1x TPU v5e-1 chip (`16 GiB` HBM):** Runs `32k` blockwise Flash-Ring Attention, `Qwen/Qwen2.5-Coder-3B-Instruct` (`128` vs `256` tiles), plus an isolated bonus `7B` probe.
- **Dual Systolic Tile Widths (`128` vs `256`):** Auto-infers `128x128` on TPU v2-v5e and `256x256` on TPU v6e Trillium / v7 Ironwood / TPU 8x via `tpu.infer_mxu_dim(device_kind)`.
- **Cache & Environment Hygiene:** Automatically picks the largest free scratch mount (`/tmp`, `/dev/shm`, `/var/tmp`) for `HF_HOME` and only installs missing build/tokenizer packages without touching preinstalled JAX or `libtpu`.

In [ ]:
# Cell 1: Select scratch mount, extract TPUForge bundle or clone repo, build C++20 runtime, probe TPU
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

best_scratch = max([p for p in ("/tmp", "/dev/shm", "/var/tmp") if os.path.isdir(p)], key=lambda p: shutil.disk_usage(p).free)
hf_cache = f"{best_scratch}/hf_cache"
os.makedirs(hf_cache, exist_ok=True)
os.environ.setdefault("HF_HOME", hf_cache)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
print(f"Selected HF_HOME={os.environ['HF_HOME']} ({shutil.disk_usage(best_scratch).free / 1e9:.1f} GB free)")

RESULTS_DIR = (
    Path("/kaggle/working/tpuforge_results")
    if Path("/kaggle/working").exists()
    else Path("./tpuforge_results")
).resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/tmp/TPUForge")
bundle_candidates = [
    Path("/tmp/tpuforge_kaggle_bundle.zip"),
    Path("/kaggle/input/tpuforge-src/tpuforge_kaggle_bundle.zip"),
    Path("/kaggle/working/tpuforge_kaggle_bundle.zip"),
    Path("./tpuforge_kaggle_bundle.zip"),
]
bundle_zip = next((p for p in bundle_candidates if p.is_file()), None)

if bundle_zip is not None:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_zip, "r") as zf:
        zf.extractall(REPO_DIR)
    print(f"Extracted TPUForge bundle from {bundle_zip} -> {REPO_DIR}")
elif Path("CMakeLists.txt").is_file() and Path("src/speculative/py_bindings.cpp").is_file():
    REPO_DIR = Path(".").resolve()
    print(f"Using current working checkout: {REPO_DIR}")
else:
    repo_url = os.environ.get("TPUFORGE_REPO_URL", "")
    if not repo_url:
        raise FileNotFoundError(
            "Upload tpuforge_kaggle_bundle.zip or set TPUFORGE_REPO_URL."
        )
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", repo_url, str(REPO_DIR)], check=True)

def _run_checked(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command exited {rc}: {' '.join(cmd)}")

missing_pkgs = []
for cli_name in ("cmake", "ninja"):
    if shutil.which(cli_name) is None:
        missing_pkgs.append(cli_name)
for mod_name in ("nanobind", "huggingface_hub", "tokenizers", "ml_dtypes"):
    if importlib.util.find_spec(mod_name) is None:
        missing_pkgs.append(mod_name)
if missing_pkgs:
    print(f"Installing missing build/runtime deps: {missing_pkgs}")
    _run_checked([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])
else:
    print("All build/runtime dependencies already present; keeping preinstalled JAX/libtpu untouched.")

build_dir = REPO_DIR / "build"
_run_checked([
    "cmake", "-S", str(REPO_DIR), "-B", str(build_dir),
    "-G", "Ninja", "-DCMAKE_BUILD_TYPE=Release",
    f"-DPython_EXECUTABLE={sys.executable}",
])
_run_checked(["cmake", "--build", str(build_dir)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(build_dir) not in sys.path:
    sys.path.insert(0, str(build_dir))

# Probe TPU hardware inside an isolated subprocess so the parent Jupyter kernel never holds /dev/vfio/*
probe_script = build_dir / "_hw_probe.py"
probe_json = RESULTS_DIR / "hw_probe.json"
probe_script.write_text(
    "import json, sys\n"
    "from pathlib import Path\n"
    f"sys.path[:0] = [{str(REPO_DIR)!r}, {str(build_dir)!r}]\n"
    "import jax\n"
    "import _tpuforge_ext as tpu\n"
    "devs = jax.devices()\n"
    "dev_kind = str(getattr(devs[0], 'device_kind', devs[0]))\n"
    "def _infer_per_device_hbm_bytes(dev, kind: str) -> int:\n"
    "    try:\n"
    "        stats = dev.memory_stats()\n"
    "        if isinstance(stats, dict) and stats.get('bytes_limit'):\n"
    "            return int(stats['bytes_limit'])\n"
    "    except Exception:\n"
    "        pass\n"
    "    low = kind.lower()\n"
    "    if 'v6' in low or 'v7' in low or 'trillium' in low or '8x' in low:\n"
    "        return 32 * (1024**3)\n"
    "    return 16 * (1024**3)\n"
    "per_dev_hbm = _infer_per_device_hbm_bytes(devs[0], dev_kind)\n"
    "total_hbm_gib = (len(devs) * per_dev_hbm) / (1024**3)\n"
    "auto_mxu = int(tpu.infer_mxu_dim(dev_kind))\n"
    "primary_model = 'Qwen/Qwen2.5-Coder-7B-Instruct' if (len(devs) >= 4 or total_hbm_gib >= 24.0) else 'Qwen/Qwen2.5-Coder-3B-Instruct'\n"
    "info = {'jax_version': jax.__version__, 'backend': jax.default_backend(), 'num_devices': len(devs), 'device_kind': dev_kind, 'total_hbm_gib': round(total_hbm_gib, 3), 'auto_mxu': auto_mxu, 'primary_model': primary_model}\n"
    f"Path({str(probe_json)!r}).write_text(json.dumps(info, indent=2), encoding='utf-8')\n"
    "print(f'JAX {jax.__version__} | backend={jax.default_backend()} | devices={len(devs)}x {dev_kind} | total HBM={total_hbm_gib:.1f} GiB | auto MXU={auto_mxu} | primary model={primary_model}')\n",
    encoding="utf-8",
)
_run_checked([sys.executable, str(probe_script)])
HW_INFO = json.loads(probe_json.read_text(encoding="utf-8"))
NUM_DEVICES = int(HW_INFO["num_devices"])
TOTAL_HBM_GIB = float(HW_INFO["total_hbm_gib"])
PRIMARY_MODEL = str(HW_INFO["primary_model"])

In [ ]:
# Cell 2: Milestone M-005 Optical Torus Ring-Attention over ICI (4k parity + 32k + 128k on 8x TPU)
import json
import sys

HW_INFO = json.loads((RESULTS_DIR / "hw_probe.json").read_text(encoding="utf-8"))
_run_checked([
    sys.executable, "-m", "benchmarks.ring_attention_jax",
    "--max-seq-len", "4096",
    "--valid-seq-len", "3072",
    "--ring-size", "auto",
    "--tp-size", "1",
    "--tile-size", "auto",
    "--check-parity",
    "--output-json", str(RESULTS_DIR / "m005_ring_4k_parity.json"),
])

_run_checked([
    sys.executable, "-m", "benchmarks.ring_attention_jax",
    "--max-seq-len", "32768",
    "--valid-seq-len", "28672",
    "--ring-size", "auto",
    "--tp-size", "1",
    "--tile-size", "auto",
    "--no-check-parity",
    "--output-json", str(RESULTS_DIR / "m005_ring_attention.json"),
])

if int(HW_INFO["num_devices"]) >= 8:
    _run_checked([
        sys.executable, "-m", "benchmarks.ring_attention_jax",
        "--max-seq-len", "131072",
        "--valid-seq-len", "114688",
        "--ring-size", "8",
        "--tp-size", "1",
        "--tile-size", "auto",
        "--no-check-parity",
        "--output-json", str(RESULTS_DIR / "m005_ring_128k.json"),
    ])
else:
    print(f"Skipping 128k 8-chip ring run: only {HW_INFO['num_devices']} device(s) visible.")

In [ ]:
# Cell 3: Hardware-Adaptive Qwen2.5-Coder Benchmark (7B on multi-chip / >=24 GiB, 3B on 16 GiB v5e-1)
import json
import sys

HW_INFO = json.loads((RESULTS_DIR / "hw_probe.json").read_text(encoding="utf-8"))
NUM_DEVICES = int(HW_INFO["num_devices"])
TOTAL_HBM_GIB = float(HW_INFO["total_hbm_gib"])
use_7b = NUM_DEVICES >= 4 or TOTAL_HBM_GIB >= 24.0
PRIMARY_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct" if use_7b else "Qwen/Qwen2.5-Coder-3B-Instruct"
primary_out = RESULTS_DIR / ("qwen_7b_bench.json" if use_7b else "qwen_3b_bench.json")
tp_arg = "auto" if NUM_DEVICES > 1 else "1"

print(f"Running Cell 3 primary benchmark: {PRIMARY_MODEL} (tp={tp_arg}, HBM={TOTAL_HBM_GIB:.1f} GiB)")
_run_checked([
    sys.executable, "-m", "benchmarks.run_qwen_bench",
    "--model", PRIMARY_MODEL,
    "--max-seq-len", "2048",
    "--max-new-tokens", "128",
    "--tile-size", "auto",
    "--tp-size", tp_arg,
    "--output-json", str(primary_out),
])

In [ ]:
# Cell 4: Systolic Tile Width Comparison (128x128 reused vs 256x256) + Single-Chip 7B Bonus Probe
import json
import os
import shutil
import subprocess
import sys

HW_INFO = json.loads((RESULTS_DIR / "hw_probe.json").read_text(encoding="utf-8"))
NUM_DEVICES = int(HW_INFO["num_devices"])
TOTAL_HBM_GIB = float(HW_INFO["total_hbm_gib"])
use_7b = NUM_DEVICES >= 4 or TOTAL_HBM_GIB >= 24.0
COMPARE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct" if use_7b else "Qwen/Qwen2.5-Coder-3B-Instruct"
tp_arg = "auto" if NUM_DEVICES > 1 else "1"
tile128_path = RESULTS_DIR / "qwen_tile_128.json"
primary_path = RESULTS_DIR / ("qwen_7b_bench.json" if use_7b else "qwen_3b_bench.json")
reports = {}

if primary_path.is_file():
    existing = json.loads(primary_path.read_text(encoding="utf-8"))
    if existing.get("tile_size") == 128:
        shutil.copyfile(primary_path, tile128_path)
        reports["128"] = existing
        print(f"Reusing {primary_path.name} as qwen_tile_128.json")

if "128" not in reports:
    _run_checked([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", COMPARE_MODEL,
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", "128",
        "--tp-size", tp_arg,
        "--output-json", str(tile128_path),
    ])
    reports["128"] = json.loads(tile128_path.read_text(encoding="utf-8"))

tile256_path = RESULTS_DIR / "qwen_tile_256.json"
_run_checked([
    sys.executable, "-m", "benchmarks.run_qwen_bench",
    "--model", COMPARE_MODEL,
    "--max-seq-len", "2048",
    "--max-new-tokens", "128",
    "--tile-size", "256",
    "--tp-size", tp_arg,
    "--output-json", str(tile256_path),
])
reports["256"] = json.loads(tile256_path.read_text(encoding="utf-8"))

print(f"\n=== Systolic Tile Comparison ({COMPARE_MODEL}, Workload 2: Code Refactor) ===")
print(f"{'Tile':<6} | {'TP':<3} | {'Vanilla tok/s':<13} | {'TPUForge tok/s':<14} | {'Tok/Pass':<8} | {'Vanilla Compiles':<16}")
for tile, rep in reports.items():
    w2 = rep["workload_2_code_refactor"]
    w1 = rep["workload_1_prompt_sweep"]
    print(
        f"{rep['tile_size']:<6} | {rep['tp_size']:<3} | "
        f"{w2['vanilla_jax']['decode_tok_per_s']:<13.2f} | "
        f"{w2['tpuforge_tree']['decode_tok_per_s']:<14.2f} | "
        f"{w2['tpuforge_tree']['mean_tokens_per_decode_pass']:<8.2f} | "
        f"{w1['vanilla_compiles']:<16}"
    )

if COMPARE_MODEL == "Qwen/Qwen2.5-Coder-3B-Instruct":
    hf_home = os.environ["HF_HOME"]
    free_bytes = shutil.disk_usage(hf_home).free
    print(f"\n[Bonus Single-Chip 7B Probe] Free space on {hf_home}: {free_bytes / (1024**3):.1f} GB")
    if free_bytes < 25 * (1024**3):
        print("Reclaiming HF_HOME cache from 3B model before attempting 7B probe...")
        shutil.rmtree(hf_home, ignore_errors=True)
        os.makedirs(hf_home, exist_ok=True)
    bonus_proc = subprocess.run([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
        "--max-seq-len", "1024",
        "--max-new-tokens", "64",
        "--tile-size", "auto",
        "--tp-size", "1",
        "--output-json", str(RESULTS_DIR / "qwen_7b_1chip_bench.json"),
    ], check=False, capture_output=True, text=True)
    print(f"7B single-chip probe exit code: {bonus_proc.returncode}")
    if bonus_proc.stdout:
        print(bonus_proc.stdout)
    if bonus_proc.returncode != 0 and bonus_proc.stderr:
        print("7B single-chip probe STDERR:\n" + bonus_proc.stderr[-3000:])

In [ ]:
# Cell 5: Qwen2.5-Coder-32B-Instruct via Megatron TP=8 across Kaggle v5e-8 (128 GB HBM pool)
import json
import os
import shutil
import subprocess
import sys

HW_INFO = json.loads((RESULTS_DIR / "hw_probe.json").read_text(encoding="utf-8"))
NUM_DEVICES = int(HW_INFO["num_devices"])
if NUM_DEVICES >= 8:
    hf_home = os.environ["HF_HOME"]
    free_bytes = shutil.disk_usage(hf_home).free
    print(f"Free space on {hf_home} before 32B: {free_bytes / (1024**3):.1f} GB")
    if free_bytes < 75 * (1024**3):
        print("Reclaiming HF_HOME cache from 7B model before downloading 32B...")
        shutil.rmtree(hf_home, ignore_errors=True)
        os.makedirs(hf_home, exist_ok=True)
        print(f"Free space after cleanup: {shutil.disk_usage(hf_home).free / (1024**3):.1f} GB")
    proc = subprocess.run([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "Qwen/Qwen2.5-Coder-32B-Instruct",
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", "auto",
        "--tp-size", "8",
        "--output-json", str(RESULTS_DIR / "qwen_32b_tp8_bench.json"),
    ], check=False, capture_output=True, text=True)
    print(f"32B TP=8 exit code: {proc.returncode}")
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0 and proc.stderr:
        print("32B STDERR:\n" + proc.stderr)
else:
    print(f"Skipping 32B TP=8 benchmark: only {NUM_DEVICES} device(s) visible.")

In [ ]:
# Cell 6: Consolidated JSON Telemetry Dump
import json
from pathlib import Path

results_dir = (
    RESULTS_DIR
    if "RESULTS_DIR" in globals()
    else (
        Path("/kaggle/working/tpuforge_results")
        if Path("/kaggle/working").exists()
        else Path("/tmp/TPUForge/tpuforge_results")
    )
)
all_reports = {}
for json_file in sorted(results_dir.glob("*.json")):
    all_reports[json_file.stem] = json.loads(json_file.read_text(encoding="utf-8"))

print("=== BEGIN_TPUFORGE_JSON ===")
print(json.dumps(all_reports, indent=2))
print("=== END_TPUFORGE_JSON ===")